# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata information
md = dataset.metadata
print(f"Dataset title: {md.name}")
print(f"Description: {md.description}")
print(f"Published: {md.datePublished}")
print(f"Cite as: {md.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are made using their `@id`.

In [ ]:
# List all record sets in the dataset
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"- {rs_id}: {getattr(rs, 'name', '')}")

# For this dataset, we assume typical usage is for a main survey record set.
# Let's list fields and columns for each record set
for rs_id in record_sets:
    print(f"\nFields and columns for record set: {rs_id}")
    rs = dataset.record_sets[rs_id]
    # List fields used by this record set
    try:
        for field in rs.fields:
            print(f"  Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    except Exception as e:
        print("  No fields or failed to retrieve fields.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we'll extract data from the available record sets.
# If you don't see any record sets above, please check the metadata for 'recordSet'.

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
    else:
        print("No records found.")

# Preview first record set with data
main_rs = record_sets[0] if record_sets else None
if main_rs and main_rs in dataframes:
    print(f"Preview for record set: {main_rs}")
    display(dataframes[main_rs].head())
else:
    print("No records to preview. Check record sets defined in the Croissant schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Replace these variables with matching @id from your data overview above (or schema documentation).
# Example IDs (change as needed for your data):
#   'cr:recordSet:survey': Main survey record set
#   'cr:field:GAD7_total': GAD-7 total field
#   'cr:field:age': Age field
#   'cr:field:gender': Gender field

from IPython.display import display

# Example: Take the first loaded record set
if len(dataframes) > 0:
    # Use main record set
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id].copy()
    print(f"Working with record set: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")

    # Try to use common fields (GAD-7, PHQ-9 or age) for numeric analysis
    # Adapt these IDs as needed to match what's printed in the overview
    possible_numeric_fields = [c for c in df.columns if 'GAD' in c or 'PHQ' in c or 'PSQ' in c or 'age' in c]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # Use first matching
        print(f"Selected numeric field for EDA: {numeric_field_id}")
    else:
        numeric_field_id = df.columns[0]  # Fallback

    # Filtering: Example, GAD7 > 10 (high anxiety scores)
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: group by gender or similar field if available
        possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'sex' in c.lower()]    
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No grouping field found (like gender/sex) in data.")
    else:
        print(f"Field '{numeric_field_id}' is not numeric, cannot EDA.")
else:
    print("No data frame loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization: Numeric field distribution and (optionally) group comparison
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Group histogram if group field available
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable data for visualization available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and records from the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.
- Explored the available record sets and fields using their `@id` references as required by the Croissant schema.
- Demonstrated loading survey records into pandas DataFrames, performing basic filtering, normalization, and group statistics.
- Provided example visualizations for main outcome scores (such as GAD-7 or PHQ-9), enabling initial analysis of mental health indicators in the Kilifi County sample.

Use this notebook as a base to further analyze additional fields, record sets, or to customize data exploration for your research questions.